# 01 — Panel Exploratory Data Analysis
Understanding the 800-patient synthetic Medicare Advantage panel.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

plt.rcParams["figure.figsize"] = (13, 5)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

import sys; sys.path.append("../src")
df = pd.read_csv("../data/raw/patient_panel_flat.csv")
print(f"Panel size: {len(df):,} patients")
df.head()


## Priority distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

colors = {"Critical": "#dc2626", "High": "#f97316", "Moderate": "#facc15", "Routine": "#4ade80"}
order = ["Critical", "High", "Moderate", "Routine"]
counts = df["outreach_priority"].value_counts().reindex(order)

axes[0].bar(counts.index, counts.values, color=[colors[p] for p in counts.index])
axes[0].set_title("Patient priority distribution", fontsize=13)
axes[0].set_ylabel("Number of patients")
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 3, f"{v} ({v/len(df)*100:.0f}%)", ha="center", fontsize=10)

risk_by_priority = df.groupby("outreach_priority")["risk_score"].mean().reindex(order)
axes[1].barh(risk_by_priority.index, risk_by_priority.values,
             color=[colors[p] for p in risk_by_priority.index])
axes[1].set_title("Average risk score by priority tier", fontsize=13)
axes[1].set_xlabel("Mean risk score (0-100)")

plt.tight_layout()
plt.savefig("../reports/figures/priority_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print(counts.to_string())


## Chronic condition burden

In [ ]:
condition_cols = [c for c in df.columns if c.startswith("has_")]
condition_names = [c.replace("has_", "").replace("_", " ") for c in condition_cols]
prevalence = df[condition_cols].mean().values * 100

fig, ax = plt.subplots(figsize=(12, 5))
sorted_idx = np.argsort(prevalence)[::-1]
ax.bar([condition_names[i] for i in sorted_idx],
       [prevalence[i] for i in sorted_idx], color="steelblue")
ax.set_ylabel("Prevalence (%)")
ax.set_title("Chronic condition prevalence across 800-patient panel", fontsize=13)
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig("../reports/figures/condition_prevalence.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Average conditions per patient: {df['n_conditions'].mean():.1f}")
print(f"Patients with 3+ conditions: {(df['n_conditions'] >= 3).sum()} ({(df['n_conditions'] >= 3).mean()*100:.0f}%)")


## Quality gaps by priority tier

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
order = ["Critical", "High", "Moderate", "Routine"]
colors_list = ["#dc2626", "#f97316", "#facc15", "#4ade80"]

gap_by_priority = df.groupby("outreach_priority")["n_quality_gaps"].mean().reindex(order)
axes[0].bar(gap_by_priority.index, gap_by_priority.values, color=colors_list)
axes[0].set_title("Average quality gaps by priority tier", fontsize=13)
axes[0].set_ylabel("Mean quality gaps per patient")

days_by_priority = df.groupby("outreach_priority")["days_since_pcp_visit"].mean().reindex(order)
axes[1].bar(days_by_priority.index, days_by_priority.values, color=colors_list)
axes[1].set_title("Days since last PCP visit by priority tier", fontsize=13)
axes[1].set_ylabel("Mean days since last visit")

plt.tight_layout()
plt.savefig("../reports/figures/gaps_by_priority.png", dpi=150, bbox_inches="tight")
plt.show()


## SDOH distribution across priority tiers

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
sdoh_cols = ["food_insecurity", "housing_instability", "transportation_barrier"]
sdoh_labels = ["Food Insecurity", "Housing Instability", "Transportation Barrier"]

for ax, col, label in zip(axes, sdoh_cols, sdoh_labels):
    pct = df.groupby("outreach_priority")[col].mean() * 100
    pct.reindex(order).plot(kind="bar", ax=ax, color=colors_list)
    ax.set_title(f"{label} rate by priority", fontsize=11)
    ax.set_ylabel("% of patients")
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.savefig("../reports/figures/sdoh_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print("Key finding: SDOH burden clusters in Critical/High tiers, compounding clinical risk.")


## Lab control rates by priority tier

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

diabetic = df[df["has_Type_2_Diabetes"] == 1]
hba1c_ctrl = (diabetic["HbA1c"] < 8.0).groupby(df["outreach_priority"]).mean() * 100
hba1c_ctrl.reindex(order).plot(kind="bar", ax=axes[0], color=colors_list)
axes[0].set_title("HbA1c controlled (<8%) — diabetics", fontsize=11)
axes[0].set_ylabel("% patients controlled")
axes[0].axhline(70, linestyle="--", color="grey", alpha=0.5, label="70% target")
axes[0].legend(fontsize=9)

htn = df[df["has_Hypertension"] == 1]
bp_ctrl = (htn["SBP"] < 130).groupby(df["outreach_priority"]).mean() * 100
bp_ctrl.reindex(order).plot(kind="bar", ax=axes[1], color=colors_list)
axes[1].set_title("BP controlled (<130 SBP) — hypertensives", fontsize=11)
axes[1].set_ylabel("% patients controlled")

dyslipid = df[df["has_Hyperlipidemia"] == 1]
ldl_ctrl = (dyslipid["LDL"] < 100).groupby(df["outreach_priority"]).mean() * 100
ldl_ctrl.reindex(order).plot(kind="bar", ax=axes[2], color=colors_list)
axes[2].set_title("LDL controlled (<100 mg/dL)", fontsize=11)
axes[2].set_ylabel("% patients controlled")

for ax in axes:
    ax.tick_params(axis="x", rotation=30)
    ax.set_xlabel("")

plt.tight_layout()
plt.savefig("../reports/figures/lab_control_rates.png", dpi=150, bbox_inches="tight")
plt.show()
